# Norwegian field and area development with NeqSim

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/fielddevelopment/norwegian_field_and_area_development_with_neqsim.ipynb)

> **FOR EDUCATION ONLY.** This notebook teaches field- and area-development methods. It is not a reserves estimate, concept-select recommendation, facility design basis, transport booking, regulatory submission, or investment decision. The technical cases and all commercial assumptions are synthetic.

The notebook connects compositional PVT, reservoir material balance, wells, SURF, a detailed NeqSim processing model, export and gas injection to annual capacity allocation, product quality, emissions, NPV and break-even analysis. It also compares greenfield and brownfield routes and demonstrates how an existing `ProcessSystem`, a multi-area `ProcessModel`, and an existing SURF system are connected to the lifecycle engine.

## Learning objectives

After completing the notebook you should be able to:

1. keep public observations separate from synthetic engineering assumptions;
2. run a reservoir-to-export annual simulation with gas recycle/injection;
3. inspect facility sizing, requested and admitted utilization, deferment and bottlenecks;
4. compare standalone, direct-host and managed-capacity tieback concepts;
5. screen area-development routes across more than one producing asset; and
6. connect a real NeqSim host plant and existing SURF model for detailed follow-up.

The workflow follows the building-block and integrated-calculation approach in the supplied NTNU input book, especially its chapters on field-development workflows, PVT, reservoirs, wells, flow assurance, subsea systems, process facilities, host capacity, economics and tiebacks.

## 1. Colab setup

While the lifecycle API is under review, this notebook builds the feature branch. After release, replace this source build with the normal released `neqsim` package installation. The build is intentionally explicit so the educational result is reproducible.

In [ ]:
# Colab/Linux setup. Run once; the Maven build can take several minutes.
!if [ ! -d /content/neqsim-src/.git ]; then git clone --depth 1 --branch agent/integrated-field-lifecycle-evaluation https://github.com/equinor/neqsim.git /content/neqsim-src; fi
!cd /content/neqsim-src && ./mvnw -B -ntp -DskipTests -Dmaven.javadoc.skip=true package
!pip install -q JPype1 pandas numpy matplotlib requests

import sys
sys.path.insert(0, "/content/neqsim-src/devtools")
from neqsim_dev_setup import neqsim_init
ns = neqsim_init("/content/neqsim-src", verbose=False)
print("NeqSim development JVM is ready")

In [ ]:
import math
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from jpype import JClass

NorwegianOilFieldCase = JClass("neqsim.process.fielddevelopment.lifecycle.NorwegianOilFieldCase")
FieldLifecycleEvaluator = JClass("neqsim.process.fielddevelopment.lifecycle.FieldLifecycleEvaluator")
AreaDevelopmentEvaluator = JClass("neqsim.process.fielddevelopment.lifecycle.AreaDevelopmentEvaluator")
FacilityModificationPlanner = JClass("neqsim.process.fielddevelopment.lifecycle.FacilityModificationPlanner")
ThermodynamicOperations = JClass("neqsim.thermodynamicoperations.ThermodynamicOperations")

plt.style.use("seaborn-v0_8-whitegrid")

## 2. Public context and source discipline

Public observations can help frame an educational area study, but they do not provide a project design basis. This notebook uses:

- [SODIR open data](https://www.sodir.no/en/facts/data-and-analyses/open-data/) and [FactPages](https://factpages.sodir.no/en) for public field, discovery, facility, pipeline and reserves context. SODIR makes downloads and a REST data service available and states the applicable open-data terms.
- The official SODIR FactPages/DataService mapping for layer identifiers: field `7100`, field reserves `7113`, discovery `7000`, facility `6000`, pipeline `6100`.
- [Gassco transport agreements](https://gassco.eu/en/shippers/transport-agreements/) and [capacity booking and reporting](https://gassco.eu/en/shippers/capacity-booking-and-reporting/) for the commercial context of Norwegian gas transport. Actual delivery-point quality, tariff, availability and booking terms must come from the applicable agreements and operator data.
- The supplied course book *Field Development and Operations: A Computational Approach with NeqSim* (Solbraa et al., 2026) as the educational method reference.

Do not silently replace missing project data with public map attributes. SODIR itself cautions that spatial data have varying accuracy and are not navigation data.

In [ ]:
# Small, reusable ArcGIS REST reader for SODIR's public DataService.
SODIR_DATA = "https://factmaps.sodir.no/api/rest/services/DataService/Data/FeatureServer"
LAYERS = {
    "facility": 6000,
    "pipeline": 6100,
    "discovery": 7000,
    "field": 7100,
    "field_reserves": 7113,
}

def sodir_layer_info(layer_id):
    response = requests.get(f"{SODIR_DATA}/{layer_id}", params={"f": "json"}, timeout=60)
    response.raise_for_status()
    payload = response.json()
    if "error" in payload:
        raise RuntimeError(payload["error"])
    return payload

def sodir_features(layer_id, where="1=1", out_fields="*", return_geometry=False,
                   return_centroid=False, record_count=2000):
    params = {
        "f": "json", "where": where, "outFields": out_fields,
        "returnGeometry": str(return_geometry).lower(), "outSR": 4326,
        "resultRecordCount": record_count,
    }
    if return_centroid:
        params["returnCentroid"] = "true"
    response = requests.get(f"{SODIR_DATA}/{layer_id}/query", params=params, timeout=60)
    response.raise_for_status()
    payload = response.json()
    if "error" in payload:
        raise RuntimeError(payload["error"])
    rows = []
    for feature in payload.get("features", []):
        row = dict(feature.get("attributes", {}))
        location = feature.get("centroid") or feature.get("geometry") or {}
        row["longitude"] = location.get("x", np.nan)
        row["latitude"] = location.get("y", np.nan)
        rows.append(row)
    return pd.DataFrame(rows)

def matching_columns(frame, *terms):
    terms = [term.lower().replace("_", "") for term in terms]
    return [column for column in frame.columns
            if all(term in column.lower().replace("_", "") for term in terms)]

In [ ]:
# Read a compact public snapshot. Network failure does not affect the synthetic NeqSim case below.
try:
    fields = sodir_features(LAYERS["field"], return_centroid=True)
    reserves = sodir_features(LAYERS["field_reserves"])
    discoveries = sodir_features(LAYERS["discovery"], return_centroid=True)
    facilities = sodir_features(LAYERS["facility"], return_geometry=True)
    pipelines = sodir_features(LAYERS["pipeline"], return_geometry=False)
    print({name: len(value) for name, value in {
        "fields": fields, "reserves": reserves, "discoveries": discoveries,
        "facilities": facilities, "pipelines": pipelines}.items()})
except Exception as error:
    print("SODIR snapshot unavailable; continuing with the synthetic teaching case:", error)
    fields = reserves = discoveries = facilities = pipelines = pd.DataFrame()

In [ ]:
# Inspect the public schema rather than assuming every field name remains unchanged.
if not fields.empty:
    preferred = [column for column in ["fldName", "fldNpdidField",
                 "fldCurrentActivitySatus", "fldHcType", "fldMainArea",
                 "longitude", "latitude"] if column in fields.columns]
    display(fields[preferred].sort_values(preferred[0]).head(20))
    print("Reserve columns containing 'remaining' and 'oil':", matching_columns(reserves, "remaining", "oil"))
else:
    display(Markdown("*No public snapshot was loaded in this run.*"))

### Data and assumption ledger

| Item | Classification in this notebook | Permitted use |
|---|---|---|
| SODIR field/discovery/facility/pipeline names and public attributes | Public observed data | Geographic and portfolio context, source demonstration |
| SODIR reserve/resource estimates | Public authority estimate | Context only; preserve edition/date and uncertainty |
| Reservoir fluid composition, volumes and PI | Synthetic educational assumption | Demonstrate PVT/material balance and sensitivity |
| Well count, SURF route, pressures and diameters | Synthetic educational assumption | Demonstrate integrated design calculations |
| Host production, uptime, holdback and capacities | Synthetic educational assumption | Demonstrate brownfield allocation and bottlenecks |
| Product limits | Synthetic placeholders | Demonstrate checks; replace with actual agreements/permits |
| Prices, tariffs, CAPEX, OPEX, tax and discount rate | Synthetic screening assumptions | Teach NPV/break-even; never an investment forecast |

A real study needs auditable asset data, uncertainty ranges, operator-validated models, applicable specifications and commercial terms.

## 3. Compositional PVT diagnostics

The reference factory creates a synthetic North Sea oil using the same thermodynamic object that feeds the reservoir and process calculations. The pressure sweep below is diagnostic, not laboratory-data regression. A project model should be tuned against separator tests, CCE/CVD/DL data, viscosity and density observations.

In [ ]:
base_fluid = NorwegianOilFieldCase.createReservoirFluid()
pvt_rows = []
for pressure_bara in np.linspace(350.0, 50.0, 25):
    fluid = base_fluid.clone()
    fluid.setTemperature(90.0, "C")
    fluid.setPressure(float(pressure_bara), "bara")
    ThermodynamicOperations(fluid).TPflash()
    fluid.initPhysicalProperties()
    gas_fraction = np.nan
    try:
        gas_fraction = sum(float(fluid.getBeta(i)) for i in range(fluid.getNumberOfPhases())
                           if "gas" in str(fluid.getPhase(i).getType()).lower())
    except Exception:
        pass
    pvt_rows.append({"pressure_bara": pressure_bara,
                     "phases": int(fluid.getNumberOfPhases()),
                     "gas_mole_fraction": gas_fraction})
pvt = pd.DataFrame(pvt_rows)
display(pvt.head())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].step(pvt.pressure_bara, pvt.phases, where="mid")
axes[0].set(xlabel="Pressure [bara]", ylabel="Number of phases", title="Synthetic PVT phase count")
axes[0].invert_xaxis()
axes[1].plot(pvt.pressure_bara, pvt.gas_mole_fraction, marker="o")
axes[1].set(xlabel="Pressure [bara]", ylabel="Gas phase mole fraction", title="Gas liberation diagnostic")
axes[1].invert_xaxis()
plt.tight_layout()

## 4. Integrated greenfield lifecycle

The executable reference case contains a material-balance reservoir, production and injection wells, subsea flowlines/risers, separation and oil stabilization, produced-water treatment, gas compression, export and gas-injection compression. The greenfield strategy sizes the detailed process at the design point, applies mechanical constraints and records train count, design power and bottleneck. Annual operating points are then constrained by reservoir deliverability, wells, nameplate capacities and the detailed process.

In [ ]:
lifecycle_evaluator = FieldLifecycleEvaluator()
greenfield_concept = NorwegianOilFieldCase.createGasInjectionCase()
greenfield = lifecycle_evaluator.evaluate(greenfield_concept)
print("Completed:", greenfield.getConceptName(), "-", greenfield.getStopReason())

In [ ]:
def lifecycle_summary(result):
    return pd.Series({
        "NPV [MUSD]": result.getNpvMusd(),
        "IRR [%]": 100.0 * result.getIrr(),
        "break-even oil [USD/bbl]": result.getBreakevenOilPriceUsdPerBbl(),
        "cumulative oil [MSm3]": result.getCumulativeOilSm3() / 1e6,
        "gas injected [GSm3]": result.getCumulativeGasInjectedSm3() / 1e9,
        "deferred oil [MSm3]": result.getCumulativeDeferredOilSm3() / 1e6,
        "peak admitted utilization [%]": 100.0 * result.getPeakFacilityUtilization(),
        "peak requested utilization [%]": 100.0 * result.getPeakUnconstrainedFacilityUtilization(),
        "off-spec years": result.getOffSpecificationYears(),
        "lifecycle CO2 [kt]": result.getLifecycleCo2Tonnes() / 1000.0,
    })
display(lifecycle_summary(greenfield).to_frame("synthetic reference"))

In [ ]:
design = greenfield.getFacilityDesignResult()
capacity = design.getNameplateCapacity()
design_table = pd.Series({
    "strategy": str(design.getStrategyName()),
    "mode": str(design.getDevelopmentMode()),
    "design oil [Sm3/d]": design.getDesignRates().getOilSm3PerDay(),
    "nameplate oil [Sm3/d]": capacity.getOilSm3PerDay(),
    "nameplate gas [Sm3/d]": capacity.getGasSm3PerDay(),
    "nameplate water [Sm3/d]": capacity.getWaterSm3PerDay(),
    "design power [kW]": design.getDesignPowerKw(),
    "auto-sized equipment": design.getAutoSizedEquipmentCount(),
    "mechanical constraints": design.getMechanicalConstraintCount(),
    "parallel trains": design.getRequiredParallelTrainCount(),
    "design bottleneck": str(design.getDesignBottleneck()),
})
display(design_table.to_frame("greenfield design"))

In [ ]:
def annual_frame(result):
    rows = []
    for item in result.getAnnualResults():
        quality = item.getProductSpecificationResult()
        rows.append({
            "year": item.getYear(), "oil_rate_sm3d": item.getAverageOilRateSm3PerDay(),
            "potential_oil_sm3d": item.getPotentialOilRateSm3PerDay(),
            "requested_oil_sm3d": item.getRequestedOilRateSm3PerDay(),
            "host_oil_sm3d": item.getHostOilRateSm3PerDay(),
            "water_cut": item.getAverageWaterCut(),
            "reservoir_pressure_bara": item.getAverageReservoirPressureBara(),
            "gas_export_msm3": item.getGasExportSm3() / 1e6,
            "gas_injected_msm3": item.getGasInjectedSm3() / 1e6,
            "deferred_oil_msm3": item.getCapacityDeferredOilSm3() / 1e6,
            "holdback_oil_msm3": item.getHoldbackOilSm3() / 1e6,
            "admitted_utilization_pct": 100.0 * item.getMaximumFacilityUtilization(),
            "requested_utilization_pct": 100.0 * item.getUnconstrainedFacilityUtilization(),
            "bottleneck": str(item.getPrimaryBottleneck()),
            "requested_bottleneck": str(item.getUnconstrainedBottleneck()),
            "co2_gas_mol_pct": quality.getGasCo2MolePercent(),
            "oil_rvp_bara": quality.getOilRvpBara(),
            "oil_in_water_mg_l": quality.getOilInWaterMgPerL(),
            "spec_violations": str(quality.getSummary()),
        })
    return pd.DataFrame(rows)

green_annual = annual_frame(greenfield)
display(green_annual.head())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
axes[0, 0].plot(green_annual.year, green_annual.potential_oil_sm3d, label="potential")
axes[0, 0].plot(green_annual.year, green_annual.oil_rate_sm3d, label="produced")
axes[0, 0].set_ylabel("Oil [Sm3/d]"); axes[0, 0].legend()
axes[0, 1].plot(green_annual.year, green_annual.reservoir_pressure_bara)
axes[0, 1].set_ylabel("Reservoir pressure [bara]")
axes[1, 0].plot(green_annual.year, green_annual.gas_export_msm3, label="export")
axes[1, 0].plot(green_annual.year, green_annual.gas_injected_msm3, label="injected")
axes[1, 0].set_ylabel("Annual gas [MSm3]"); axes[1, 0].legend()
axes[1, 1].plot(green_annual.year, 100.0 * green_annual.water_cut)
axes[1, 1].set_ylabel("Water cut [%]")
for axis in axes[1, :]: axis.set_xlabel("Year")
fig.suptitle("Synthetic gas-injection reference case")
plt.tight_layout()

## 5. Annual bottlenecks and deferment

`requested utilization` is the load before capacity allocation; it can exceed 100% and reveals the size of the constraint. `admitted utilization` is the load actually sent through the plant after holdback and capacity allocation. Reporting only admitted utilization would hide rejected production and understate the modification need.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(green_annual.year, green_annual.requested_utilization_pct, label="requested")
ax.plot(green_annual.year, green_annual.admitted_utilization_pct, label="admitted")
ax.axhline(100.0, color="black", linestyle="--", linewidth=1)
ax.set(xlabel="Year", ylabel="Maximum facility utilization [%]")
ax.legend()
display(green_annual[["year", "requested_bottleneck", "requested_utilization_pct",
                      "deferred_oil_msm3"]].sort_values("requested_utilization_pct", ascending=False).head(10))

In [ ]:
quality_columns = ["year", "co2_gas_mol_pct", "oil_rvp_bara",
                   "oil_in_water_mg_l", "spec_violations"]
display(green_annual[quality_columns])
print("Lifecycle product summary:", greenfield.getProductSpecificationSummary())

## 6. Concept comparison: injection, depletion and brownfield hosts

The factory portfolio holds the teaching assumptions consistent while changing the development route and operating policy. Brownfield cases include declining host production. Host-priority and proportional policies demonstrate capacity utilization, host/satellite holdback and new-field deferment. Results are ranked by after-tax NPV, but engineering feasibility and specification eligibility remain explicit gates.

In [ ]:
portfolio_results = lifecycle_evaluator.evaluateAll(NorwegianOilFieldCase.createDevelopmentPortfolio())
display(Markdown(str(lifecycle_evaluator.toMarkdownTable(portfolio_results))))
comparison = pd.DataFrame({str(result.getConceptName()): lifecycle_summary(result)
                           for result in portfolio_results}).T
display(comparison)

In [ ]:
# Compare admitted and requested host utilization through time.
fig, ax = plt.subplots(figsize=(12, 5))
for result in portfolio_results:
    if "tieback" in str(result.getConceptName()).lower():
        frame = annual_frame(result)
        ax.plot(frame.year, frame.requested_utilization_pct, label=str(result.getConceptName()))
ax.axhline(100.0, color="black", linestyle="--", linewidth=1)
ax.set(xlabel="Year", ylabel="Requested utilization [%]", title="Brownfield host loading before allocation")
ax.legend(fontsize=8)

## 7. Area-development routing

An area portfolio attaches route metadata to full lifecycle concepts. The example compares a standalone FPSO, two policies at host A and a longer route to host B. In a real study, populate candidate hosts from validated operator data; use SODIR only to create a public-context long list. Distances, tie-in points, hydraulic models, shutdown windows and ownership/commercial terms require confirmation.

In [ ]:
area_result = AreaDevelopmentEvaluator().evaluate(NorwegianOilFieldCase.createAreaDevelopmentPortfolio())
display(Markdown(str(area_result.toMarkdownTable())))
recommended = area_result.getRecommendedOption()
print("Highest-NPV eligible teaching option:",
      None if recommended is None else recommended.getOption().getName())

In [ ]:
area_rows = []
for option_result in area_result.getRankedOptions():
    option = option_result.getOption()
    result = option_result.getLifecycleResult()
    area_rows.append({"option": str(option.getName()), "route": str(option.getRouteType()),
                      "receiving_asset": str(option.getReceivingAssetName()),
                      "eligible": bool(option_result.isEligible()),
                      "npv_musd": result.getNpvMusd(),
                      "break_even_usd_bbl": result.getBreakevenOilPriceUsdPerBbl(),
                      "deferred_oil_msm3": result.getCumulativeDeferredOilSm3() / 1e6,
                      "peak_requested_utilization_pct":
                          100.0 * result.getPeakUnconstrainedFacilityUtilization()})
area_table = pd.DataFrame(area_rows)
display(area_table)
area_table.plot.barh(x="option", y="npv_musd", legend=False, title="Synthetic option NPV [MUSD]")

## 8. Debottleneck and modification screening

The planner converts years above a target utilization into traceable screening actions. Detailed equipment names are preserved. SURF, flowline, pipeline, riser and subsea bottlenecks lead to SURF rerating/looping/new-riser/alternate-tie-in suggestions; processing constraints lead to relevant train or equipment review. These are study candidates, not modification designs.

In [ ]:
host_a = lifecycle_evaluator.evaluate(NorwegianOilFieldCase.createHostPriorityTiebackCase())
modification_plan = FacilityModificationPlanner().analyse(host_a, 0.90)
display(Markdown(str(modification_plan.toMarkdownTable())))

## 9. Use an existing detailed brownfield `ProcessSystem`

The teaching factory is replaceable. A real host study should supply the operator's detailed NeqSim flowsheet, with equipment mechanical limits, control assumptions, host feeds and analyzers. The lifecycle builder maps connection points rather than rebuilding the plant.

```java
FieldLifecycleModel brownfield = FieldLifecycleModel
    .existingFacility("Satellite to real host", satelliteReservoir, existingHostProcessSystem)
    .reservoirStreams(satelliteOilProducer, satelliteWaterProducer, satelliteGasInjector)
    .gasHandling(recoveredGas, exportInjectionSplitter, compressedInjectionGas)
    .exportStreams(hostStabilizedOil, hostSalesGas)
    .hostFeeds(existingHostOilFeed, existingHostGasFeed, existingHostWaterFeed)
    .treatedWaterDischarge(hostTreatedWater)
    .productQualityProvider(realPlantAnalyser)
    .build();
```

The exact process system is executed at every admitted annual operating point. NeqSim's equipment sizing/mechanical constraints, process power and bottleneck calculations are therefore retained. Host production and allocation policy are provided through `HostFacility` and `FacilityLifecycleStrategy.brownfield(...)`.

## 10. Use a multi-area `ProcessModel` and an existing SURF system

SURF can be a separate connected `ProcessSystem` or a named area in an existing `ProcessModel`. The two systems must already share the connecting NeqSim stream topology. The complete model is solved; power, sizing and bottleneck scans cover all areas, and bottlenecks are reported as `area::equipment`.

```java
ProcessModel infrastructure = new ProcessModel();
infrastructure.add("shared SURF", existingSurfProcessSystem);
infrastructure.add("host topsides", existingHostProcessSystem);

FieldLifecycleModel route = FieldLifecycleModel
    .existingSurfAndFacility("Discovery via shared SURF", satelliteReservoir,
        infrastructure, "shared SURF", "host topsides")
    .reservoirStreams(satelliteOilProducer, satelliteWaterProducer, satelliteGasInjector)
    .gasHandling(recoveredGas, exportInjectionSplitter, compressedInjectionGas)
    .exportStreams(hostStabilizedOil, hostSalesGas)
    .hostFeeds(existingHostOilFeed, existingHostGasFeed, existingHostWaterFeed)
    .treatedWaterDischarge(hostTreatedWater)
    .build();
```

For two already-connected systems, the convenience overload is:

```java
FieldLifecycleModel.existingSurfAndFacility(
    "Discovery via inherited route", reservoir, existingSurfSystem, existingFacilitySystem);
```

This is the key route for testing whether a new field can reuse an existing manifold, flowline, riser and host plant, and for identifying whether the modification belongs subsea or topsides.

## 11. Product specifications and gas transport

The lifecycle check supports gas CO2, H2S, oxygen, water and hydrocarbon dew points, gross calorific value, Wobbe index and relative density; oil RVP and BS&W; and oil-in-water. A specification policy can warn, curtail, or reject an area option.

The numerical limits in the factory are synthetic. For a Norwegian project, configure limits and measurement bases from the actual sales agreement, Gassco entry/exit point terms, liquid export agreement and discharge permit. Treat transport capacity as a time-dependent commercial constraint: distinguish firm and interruptible capacity, booked quantities and tariffs. [Gassco's official agreement page](https://gassco.eu/en/shippers/transport-agreements/) and [booking page](https://gassco.eu/en/shippers/capacity-booking-and-reporting/) are starting points, not substitutes for the applicable contract.

## 12. Economics and uncertainty

NPV, IRR, payback and oil/gas break-even are calculated from annual oil and gas sales, tariffs, fixed/variable OPEX, CAPEX timing and the configured Norwegian tax model. The break-even is a root solve on the same integrated production schedule.

For a real decision, create immutable configurations for low/base/high subsurface, uptime, host decline, facility availability, start date, CAPEX, OPEX, prices, tariffs, CO2 cost, tax and discount rate. Do not change one case after simulation and call it a sensitivity: build and rerun independent models because reservoir and process objects are stateful. Probabilistic aggregation and decision analysis are follow-on work, not implied by a single deterministic NPV.

In [ ]:
# Educational validation gates: fail loudly if the integrated result is internally inconsistent.
assert len(greenfield.getAnnualResults()) > 0
assert greenfield.getCumulativeOilSm3() > 0.0
assert greenfield.getFinalReservoirPressureBara() < greenfield.getInitialReservoirPressureBara()
assert greenfield.getFacilityDesignResult().getRequiredParallelTrainCount() >= 1
assert greenfield.getPeakUnconstrainedFacilityUtilization() >= greenfield.getPeakFacilityUtilization()
assert len(area_result.getRankedOptions()) >= 4
assert area_result.getRecommendedOption() is not None
print("Integrated educational validation gates passed")

## 13. Suggested exercises

1. Select a public SODIR discovery and create a provenance table containing only observed attributes; list every missing project input separately.
2. Replace the synthetic fluid with a characterized laboratory PVT model and document regression quality.
3. Add a detailed well/network production-potential provider and compare its schedule with the screening PI model.
4. Import an existing host `ProcessSystem`, then add real host-production profiles and equipment constraints.
5. Put a shared manifold, flowline and riser in a SURF area of `ProcessModel`; identify the first year and exact equipment causing deferment.
6. Compare host-priority, pro-rata, planned satellite holdback and interruptible gas-export capacity.
7. Create modification cases and rerun the complete lifecycle rather than crediting all deferred oil automatically.
8. Run low/base/high cases for subsurface, uptime, start date and cost; explain whether the ranking is robust.

### Final reminder

This notebook is solely for education about field and area development. Its synthetic outputs must not be quoted as facts about any Norwegian field, host, discovery, facility, transport route or commercial agreement.